[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/01_calibration_and_platt_scaling.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Calibration: Platt, isotonic and prior shift

Allow about 35 minutes. Can you interpret a score as a probability of AF, and which
recalibration method should you trust? We will construct a calibration report using
library functions, fit two recalibration maps on the calibration split, fix a
comparison that was fitted on the wrong recordings, and adapt a calibrated model to
a new prevalence.

The action is **send this recording for AF review**, priced in reviews as in the
core notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0. Please
attempt each problem on your own. Please replace each `...` and run the checker
below it. If you get stuck for about 10 minutes, you can use the stuck switch below
the problem, where there is one, or open the worked copy in `extension/solutions/`.
Note that some later cells rely on an earlier problem: if that problem is
unfinished, the cell prints `Waiting for ...`. To resolve this, once the problem is
done or switched, click in the waiting cell, choose **Runtime > Run before**, then
run the waiting cell.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit, logit
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
import statsmodels.api as sm

from clinical_utility import calibration as course_calibration, data, reference
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
scores = data.load_scores()
development = scores[scores["split"] == "development"]
calibration_split = scores[scores["split"] == "calibration"]
evaluation = scores[scores["split"] == "evaluation"]

## What a calibration report contains

**The Brier score** is the mean squared error between the stated probability and the
outcome (one for AF, zero otherwise) over the set of recordings. For the core's six
development recordings and the random forest's scores, this is 0.322. The Brier
score is sensitive to both discriminative ability (separating AF from non-AF) and
calibration (stating the right probability), and so is not a pure measure of
calibration.

**A reliability plot** groups the recordings by score into equal-count bins and then
measures the fraction with AF in each. It plots the mean score for each bin on the
x-axis against the fraction with AF on the y-axis. If a point is below the diagonal,
the scores in that bin are too high. We can extract the points with
`calibration_curve(y, p, n_bins=10, strategy="quantile")`, which returns them as
`(prob_true, prob_pred)`, with the observed AF fraction as the first element.

**The calibration intercept and slope** are estimated on the log-odds scale. The
logit converts a probability to its log-odds, `logit(p) = log(p / (1 - p))`. The
inverse of the logit is the sigmoid function, `expit(z) = 1 / (1 + exp(-z))`, which
converts log-odds back to a probability. For example, a score of 0.8 would have odds
of 4 and so a logit of 1.386.

- Calibration-in-the-large is a model with the slope set to one and only the
  intercept fit, with `logit(score)` as a fixed offset:
  `logit P(AF) = a + logit(score)`. In statsmodels this is a binomial GLM with the
  design matrix as a column of ones and `offset=logit(p)`. The ideal value for this
  is zero; a negative `a` indicates the scores systematically overestimate risk.
- The calibration slope is a second model with both the slope and intercept fit:
  `logit P(AF) = a + b * logit(score)`, a binomial GLM on
  `sm.add_constant(logit(p))`. The ideal value for this is one; values below one
  indicate that the log-odds are too extreme. Only the value of `b` is considered
  from this fit.

### Problem A: Make

Please implement the `calibration_report(y, p)` function. This should return a
dictionary with the following keys: `brier` (the `brier_score_loss`), `observed` and
`predicted` (the two arrays from `calibration_curve` with ten quantile bins), as
well as `intercept` and `slope` (the two GLM fits above, both with
`family=sm.families.Binomial()`).

In [ ]:
def calibration_report(y, p):
    y = np.asarray(y, dtype=float)
    z = logit(np.asarray(p, dtype=float))
    observed, predicted = calibration_curve(y, p, n_bins=10, strategy="quantile")
    in_the_large = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=z).fit()
    slope = sm.GLM(y, sm.add_constant(z), family=sm.families.Binomial()).fit()
    return {"brier": brier_score_loss(y, p), "observed": observed, "predicted": predicted,
            "intercept": float(np.asarray(in_the_large.params)[0]),
            "slope": float(np.asarray(slope.params)[1])}
_ = check.run("cal_report", calibration_report)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# calibration_report = check.use_reference("calibration_report")

In [ ]:
if check.ready(calibration_report=calibration_report):
    reports = {name: calibration_report(development["af"], development[column])
               for name, column in (("Random forest", "random_forest"), ("Raw ResNet", "resnet"))}
    display(pd.DataFrame({name: {k: r[k] for k in ("brier", "intercept", "slope")}
                          for name, r in reports.items()}).round(3))
    display(view.reliability({name: (r["observed"], r["predicted"]) for name, r in reports.items()}))

Read the raw ResNet's column. It has a calibration intercept of -2.70 on
development, indicating its scores are too high overall, and a slope of 0.74, which
indicates they are too extreme. In the top decile of its development scores, the
average score is 0.944; however, only 48.7% of those recordings actually have AF.
The random forest has a calibration intercept and slope of -0.04 and 1.04
respectively, close to their ideals. A limitation of this is that we've only used a
handful of bins and two coefficients, and these do not establish calibration in
every subgroup or in a new population.

## Fit a recalibration map on the calibration split

Platt scaling recalibrates a model by mapping a score to
`expit(a + b * logit(score))`, where `a` and `b` are learned by a logistic
regression of the AF label on `logit(score)`. It is fitted on the calibration split,
as that split is not used to choose policies (development) or to assess them
(evaluation).

### Problem B: Make

Please construct the `platt_map` model. This must be a pipeline of
`FunctionTransformer(logit)` and
`LogisticRegression(C=np.inf, solver="newton-cholesky")`, fitted on
`calibration_split[["resnet"]]` and `calibration_split["af"]`. Note that `C=np.inf`
switches off the penalty that is applied by default. If you had the fitted
classifier itself, you would be able to use `CalibratedClassifierCV` to do the same
job (Extension 3 uses it); here, however, you only have the exported scores, and so
need to fit the map directly.

In [ ]:
platt_map = make_pipeline(FunctionTransformer(logit),
                          LogisticRegression(C=np.inf, solver="newton-cholesky")).fit(
    calibration_split[["resnet"]], calibration_split["af"])
_ = check.run("cal_platt", platt_map)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# platt_map = check.use_reference("platt_map")

In [ ]:
if check.ready(platt_map=platt_map):
    y = development["af"]
    raw = development["resnet"]
    mapped = platt_map.predict_proba(development[["resnet"]])[:, 1]
    rows = {}
    for label, p in (("raw ResNet", raw), ("Platt", mapped)):
        best = reference.cheapest(reference.operating_points(y, p, costs))
        stated = reference.operating_summary(y, p >= 0.2)
        rows[label] = {"AUC": roc_auc_score(y, p), "best cutoff": best["threshold"],
                       "reviews at best cutoff": best["reviews"],
                       "AF missed at best cutoff": best["FN"],
                       "reviews at 0.2": stated["reviews"], "AF missed at 0.2": stated["af_missed"]}
    display(pd.DataFrame(rows).round(3))

A recalibration map with a positive slope preserves the ordering of the scores (and
so any ties); the AUC is therefore identical, and the best achievable policy is the
same: on development, both reach their minimum loss with 677 reviews and 49 AF
missed (at different cutoffs). The difference is the score that identifies each
decision, and so what acting at a stated 0.2 does: the raw scores send 1,051
development recordings for review, and the recalibrated ones 739.

### Problem C: Make

Unlike Platt scaling, which fits a curve, isotonic regression fits a non-decreasing
function: a series of flat blocks where the output is constant, which scikit-learn's
`predict` joins by linear interpolation. Please construct
`isotonic_map = IsotonicRegression(out_of_bounds="clip")` and fit it on the
calibration split's `resnet` scores and `af` labels. Note that
`out_of_bounds="clip"` gives a score outside the calibration range the value at the
nearest end, instead of NaN.

In [ ]:
isotonic_map = IsotonicRegression(out_of_bounds="clip").fit(calibration_split["resnet"],
                                                            calibration_split["af"])
_ = check.run("cal_isotonic", isotonic_map)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# isotonic_map = check.use_reference("isotonic_map")

In [ ]:
# The three versions of the ResNet's scores on the development split.
if check.ready(platt_map=platt_map, isotonic_map=isotonic_map):
    y = development["af"]
    versions = {"raw": development["resnet"],
                "Platt": platt_map.predict_proba(development[["resnet"]])[:, 1],
                "isotonic": isotonic_map.predict(development["resnet"])}
    rows = {}
    for label, p in versions.items():
        brier = brier_score_loss(y, p)
        fit = course_calibration.intercept_slope(y, p)
        summary = reference.operating_summary(y, p >= 0.2)
        rows[label] = {"Brier": brier, "intercept": fit["intercept"], "slope": fit["slope"],
                       "reviews per 1,000 at 0.2": 1000 * summary["reviews"] / len(y),
                       "AF missed per 1,000 at 0.2": 1000 * summary["af_missed"] / len(y)}
        for reason in fit["reasons"]:
            print(f"{label}: intercept or slope not available, because {reason}.")
    display(pd.DataFrame(rows).round(3))
    print("Fitted isotonic levels:", len(np.unique(isotonic_map.y_thresholds_)))

Read the development table, the split on which policies are chosen. The isotonic map
has a slightly higher Brier score than the Platt map (0.0565 vs 0.0559). Note that
the intercept and slope cannot be estimated for the isotonic map, as it outputs
exactly zero for the lowest scores (and can output one), where the log-odds are
infinite. This does not affect the Brier score, reliability or decisions. We can see
that the isotonic map merges many different raw scores into 15 flat fitted blocks,
with linear interpolation between them. This gives 42 distinct values on the
evaluation split, so several cutoffs that the raw scores could separate collapse
into one. A flexible map needs more calibration data than a two-parameter one.

### Problem D: Fix the bug

A colleague wrote the function below to choose between the two maps, and determined
that isotonic regression calibrates better. Based on this finding, the service
adopted the isotonic map. Your task is to fix `compare_recalibrations` in place so
that the two maps are fairly compared: fit both maps on the calibration split and
compare them on development, where policies are chosen.

In [ ]:
def compare_recalibrations(frame):
    """Brier scores of the Platt and isotonic recalibrations of the raw ResNet, to choose between them."""
    fitting = frame[frame["split"] == "calibration"]
    assessed = frame[frame["split"] == "development"]
    platt = make_pipeline(FunctionTransformer(logit),
                          LogisticRegression(C=np.inf, solver="newton-cholesky"))
    platt.fit(fitting[["resnet"]], fitting["af"])
    isotonic = IsotonicRegression(out_of_bounds="clip").fit(fitting["resnet"], fitting["af"])
    return {"platt": brier_score_loss(assessed["af"], platt.predict_proba(assessed[["resnet"]])[:, 1]),
            "isotonic": brier_score_loss(assessed["af"], isotonic.predict(assessed["resnet"]))}


_ = check.show(compare_recalibrations, scores)
_ = check.run("cal_leak", compare_recalibrations)

In [ ]:
# Choose on development, then report the frozen choice on evaluation.
if check.ready(platt_map=platt_map, isotonic_map=isotonic_map):
    difference = course_calibration.brier_difference(
        development["af"], platt_map.predict_proba(development[["resnet"]])[:, 1],
        isotonic_map.predict(development["resnet"]))
    print("Development Brier, isotonic minus Platt:", round(difference["estimate"], 5),
          "(95% interval", round(difference["low"], 5), "to", round(difference["high"], 5), ")")
    chosen = "Platt" if difference["estimate"] >= 0 else "isotonic"
    p = (platt_map.predict_proba(evaluation[["resnet"]])[:, 1] if chosen == "Platt"
         else isotonic_map.predict(evaluation["resnet"]))
    fit = course_calibration.intercept_slope(evaluation["af"], p)
    summary = reference.operating_summary(evaluation["af"], p >= 0.2)
    print(f"Frozen choice: {chosen}. On evaluation: Brier {brier_score_loss(evaluation['af'], p):.4f}, "
          f"intercept {fit['intercept']}, slope {fit['slope']}, "
          f"reviews per 1,000 at 0.2 {1000 * summary['reviews'] / len(p):.1f}, "
          f"AF missed per 1,000 {1000 * summary['af_missed'] / len(p):.1f}")

#### What the leak did

Fitting on the evaluation recordings means that both maps are judged on the outcomes
they were fitted to. The two-parameter Platt map does not benefit much from this:
its Brier score is 0.0575 either way. The flexible isotonic map, however, benefits
substantially, with a Brier score of 0.0559 when fitted on evaluation compared to
0.0583 when fitted on the calibration split. This made the isotonic map seem better
than the Platt map. Compared honestly on development, isotonic's Brier score is
higher by 0.00057 (95% paired interval -0.00024 to 0.00146). As the two maps are
close and the interval includes zero, the simpler Platt map is frozen. Its Brier
score on evaluation is 0.0575.

## Transfer the prior, not the ranking

Given prior shift, the distribution of scores within each reference class remains
the same and only the prevalence changes. We can use Bayes' rule to determine how
the probabilities should change for a different prevalence, by multiplying the odds
by the ratio of the new prior odds to the old prior odds. On the log-odds scale,
this is equivalent to an intercept shift:
`logit(p_new) = logit(p_old) + logit(prevalence_new) - logit(prevalence_old)`.

### Problem E: Make

Implement `shift_prior` using the `logit` and `expit` functions from scipy. Below,
we can see the effect of the shift when the evaluation recordings are resampled to a
prevalence of 2%. Note that the old prevalence is the AF fraction of the split on
which the map was fitted (the calibration split), not the mean score in the new
population. Also note that nothing is fitted on the resampled outcomes.

In [ ]:
def shift_prior(p, old_prevalence, new_prevalence):
    return expit(logit(p) + logit(new_prevalence) - logit(old_prevalence))
_ = check.run("cal_shift", shift_prior)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# shift_prior = check.use_reference("shift_prior")

In [ ]:
y_eval = evaluation["af"].to_numpy()
negative, positive = np.flatnonzero(~y_eval), np.flatnonzero(y_eval)
target = 0.02
keep = int(round(len(negative) * target / (1 - target)))
chosen = np.random.default_rng(20261106).choice(positive, keep, replace=False)
rows = np.r_[negative, np.sort(chosen)]
if check.ready(shift_prior=shift_prior):
    p = evaluation["resnet_calibrated"].to_numpy()[rows]
    truth = y_eval[rows]
    shifted = shift_prior(p, calibration_split["af"].mean(), target)
    table = {}
    for label, q in (("before the shift", p), ("after the shift", shifted)):
        review = q >= 0.2
        table[label] = {"mean probability": q.mean(), "observed AF fraction": truth.mean(),
                        "reviews per 1,000 at 0.2": 1000 * review.mean(),
                        "AF detected per 1,000": 1000 * np.mean(review & truth),
                        "AF missed per 1,000": 1000 * np.mean(~review & truth)}
    display(pd.DataFrame(table).round(4))

We can see that the mean probability is 0.0699 and is shifted to 0.0201 (the
observed prevalence is 0.0202). If we act at a stated 0.2, the shifted probabilities
review only 0.7 recordings per 1,000, compared to 145.3, and miss 20.2 AF per 1,000,
compared to 1.5. Note that the threshold has not changed; however, the same score
now means a lower AF probability, and so almost no recording reaches it. Given these
prices and this prevalence, the consistent policy with this model is to review
almost no one and accept that almost all AF will be missed. A service that finds
itself in this position needs either a model that separates the two classes better,
or to revisit the price of a miss. It's important to note that the agreement of the
means checks only this prior-shift scenario. If the data change in a different way,
for example a new device, more noise or a different mix of N and O rhythms, the
distribution of scores within each class can change, and then just modifying the
intercept is not enough.

## Check your report

- Which recordings were used for fitting each map and which for assessing it?
- If a map has a positive slope, why does this not affect the AUC or the best
  achievable policy, but change what a stated 0.2 does?
- When is it not possible to calculate a calibration intercept and slope and what
  should be reported instead?
- What is the assumption made by the prior shift?

**Question to consider:** if the calibration slope also changes in a new service,
what evidence would you need to see to be happy with just an intercept-only update?